# Healthcare Natural Language → SQL Agent
### Querying Texas Medicare Billing Data (CMS 2023) with Plain English using Claude AI

**Author:** Hassaan Ahmed Siddiqui — BI Analyst  
**Data:** CMS Medicare Physician & Other Practitioners by Provider and Service (2023)  
**Scope:** 671,400 Texas provider-procedure records | $10B+ in Medicare billing activity  

---

## What this notebook does
1. **Loads** real CMS Medicare billing data for Texas providers (2023)
2. **Transforms** it into a fast SQLite database with revenue cycle metrics
3. **Builds** an AI agent using Claude that converts plain English into SQL
4. **Demonstrates** live queries against real healthcare billing data

---

## Section 1 — Data Discovery & Setup

### Import Libraries
We start by importing the Python libraries needed for this project:
- **pandas** — reads and processes the large CSV file
- **sqlite3** — creates and queries the database
- **os** — checks if files exist on disk
- **glob** — finds files by pattern (e.g. any file starting with MUP_PHY_)

In [ ]:
import pandas as pd
import sqlite3
import os
import glob

print("✓ All libraries imported successfully")
print(f"  pandas version : {pd.__version__}")
print(f"  sqlite3 version: {sqlite3.sqlite_version}")

### Locate the CMS Data File
Before reading 2GB of data, we first confirm Python can see the file.
This catches the most common beginner mistake — running Python from the wrong folder.
We also print the file size to confirm the full file downloaded correctly.

In [ ]:
# Automatically find the CMS CSV in your folder
matches = glob.glob("MUP_PHY_*.csv") or glob.glob("*.csv")

if not matches:
    print("ERROR: No CSV found. Make sure you are in the nl-sql-agent folder.")
else:
    csv_path = sorted(matches, key=len, reverse=True)[0]
    file_size_gb = os.path.getsize(csv_path) / (1024**3)
    print(f"✓ Found file : {csv_path}")
    print(f"  File size  : {file_size_gb:.2f} GB")

### Explore the Raw Dataset
We read just 3 rows of the file to inspect its structure without loading the full 2GB into memory.
This is a standard first step in any data project — understand what you have before you transform it.
The CMS file has 28 columns. We will select only the 15 relevant to revenue cycle analysis.

In [ ]:
# Read just 3 rows — fast, no waiting
preview = pd.read_csv(csv_path, nrows=3)

print(f"Total columns in file: {len(preview.columns)}")
print(f"\nColumn names:")
for i, col in enumerate(preview.columns, 1):
    print(f"  {i:>2}. {col}")

### Select Columns & Define Configuration
Out of 28 columns in the raw file, we keep only 15 that are relevant to healthcare revenue cycle analysis.
Dropping unnecessary columns reduces memory usage and keeps the database focused.
We also define key configuration variables here so they are easy to change in one place:
- **STATE_FILTER** — change 'TX' to any state abbreviation to analyse a different state
- **DB_PATH** — the name of the SQLite database file we will create
- **CHUNK_SIZE** — how many rows to read at a time (protects against memory overflow)

In [ ]:
# The columns we need for revenue cycle analysis
COLUMNS_TO_KEEP = [
    "Rndrng_NPI",                    # Provider unique ID
    "Rndrng_Prvdr_Last_Org_Name",    # Provider/practice name
    "Rndrng_Prvdr_First_Name",       # First name
    "Rndrng_Prvdr_Type",             # Specialty (Cardiology, Internal Medicine...)
    "Rndrng_Prvdr_City",             # City
    "Rndrng_Prvdr_State_Abrvtn",     # State abbreviation (TX, CA...)
    "Rndrng_Prvdr_Zip5",             # ZIP code
    "HCPCS_Cd",                      # Procedure billing code
    "HCPCS_Desc",                    # Plain English procedure name
    "Place_Of_Srvc",                 # F = facility, O = office
    "Tot_Benes",                     # Total Medicare beneficiaries
    "Tot_Srvcs",                     # Total services billed
    "Avg_Sbmtd_Chrg",                # Average amount charged per service
    "Avg_Mdcr_Alowd_Amt",            # Average Medicare allowed per service
    "Avg_Mdcr_Pymt_Amt",             # Average Medicare payment per service
]

STATE_FILTER = "TX"        # Filtering to Texas
DB_PATH      = "cms_texas.db"
CHUNK_SIZE   = 100_000

print(f"✓ Keeping {len(COLUMNS_TO_KEEP)} of {len(preview.columns)} columns")
print(f"  Filtering to : {STATE_FILTER} providers only")
print(f"  Output file  : {DB_PATH}")

### Load, Filter & Transform Data
This is the core ETL (Extract, Transform, Load) step.

**Extract** — reads the 2GB CSV file in chunks of 100,000 rows at a time.
Reading in chunks means this works even on a laptop with limited RAM —
we never load the full file into memory at once.

**Transform** — for each chunk we:
1. Filter to Texas providers only (reduces 9M rows to ~671,400)
2. Convert text columns to numbers for calculation
3. Derive total dollar amounts by multiplying averages by number of services
4. Calculate two key revenue cycle metrics:
   - **Reimbursement Rate %** — what % of billed amount Medicare actually paid
   - **Revenue Leakage** — dollars billed but never collected from Medicare

**Load** — writes each filtered chunk into a SQLite database called cms_texas.db

In [ ]:
# Remove old database if re-running
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("Removed old database — starting fresh\n")

conn       = sqlite3.connect(DB_PATH)
total_rows = 0
chunk_num  = 0

for chunk in pd.read_csv(
    csv_path,
    usecols=COLUMNS_TO_KEEP,
    chunksize=CHUNK_SIZE,
    dtype=str,
    low_memory=False
):
    chunk_num += 1

    # Filter to Texas only
    filtered = chunk[chunk["Rndrng_Prvdr_State_Abrvtn"] == STATE_FILTER].copy()

    if filtered.empty:
        continue

    # Convert numeric columns from string to float
    for col in ["Tot_Benes", "Tot_Srvcs", "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Pymt_Amt"]:
        filtered[col] = pd.to_numeric(filtered[col], errors="coerce")

    # Derive total dollar amounts (avg × number of services)
    filtered["Tot_Sbmtd_Chrg"]     = (filtered["Avg_Sbmtd_Chrg"]     * filtered["Tot_Srvcs"]).round(2)
    filtered["Tot_Mdcr_Alowd_Amt"] = (filtered["Avg_Mdcr_Alowd_Amt"] * filtered["Tot_Srvcs"]).round(2)
    filtered["Tot_Mdcr_Pymt_Amt"]  = (filtered["Avg_Mdcr_Pymt_Amt"]  * filtered["Tot_Srvcs"]).round(2)

    # Revenue cycle metrics
    filtered["Reimbursement_Rate_Pct"] = (
        filtered["Tot_Mdcr_Pymt_Amt"] / filtered["Tot_Sbmtd_Chrg"] * 100
    ).round(2)

    filtered["Revenue_Leakage"] = (
        filtered["Tot_Sbmtd_Chrg"] - filtered["Tot_Mdcr_Pymt_Amt"]
    ).round(2)

    # Write to SQLite
    filtered.to_sql(
        "cms_billing",
        conn,
        if_exists="append" if total_rows > 0 else "replace",
        index=False
    )

    total_rows += len(filtered)
    print(f"  Chunk {chunk_num:>3} — added {len(filtered):>6,} TX rows  |  total: {total_rows:>8,}")

print(f"\n✓ Done loading. Total Texas rows: {total_rows:,}")

### Create Database Indexes
Indexes work exactly like the index at the back of a textbook.
Instead of scanning all 671,400 rows for every query, SQLite jumps directly to relevant rows.

We create 5 indexes on the columns most likely to appear in WHERE clauses:
- State, Specialty, City, Procedure Code, Provider ID

**Without indexes:** queries take 3-5 seconds
**With indexes:** queries run in milliseconds

This matters for the AI agent demo — instant responses feel like magic,
slow responses lose the audience.

In [ ]:
# Creating index for query performance
cur = conn.cursor()

cur.execute("CREATE INDEX IF NOT EXISTS idx_state  ON cms_billing (Rndrng_Prvdr_State_Abrvtn)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_type   ON cms_billing (Rndrng_Prvdr_Type)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_city   ON cms_billing (Rndrng_Prvdr_City)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_hcpcs  ON cms_billing (HCPCS_Cd)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_npi    ON cms_billing (Rndrng_NPI)")

conn.commit()
print("✓ Indexes created — queries will now run in milliseconds")

### Validate & Summarise the Database
Before building the AI agent on top of this data, we validate what we loaded.
This serves two purposes:
1. **Quality check** — confirms the row counts, column values, and dollar figures look reasonable
2. **Business insight** — the summary already tells a story about Texas Medicare billing
   that a hospital CFO would find immediately useful

The gap between Total Billed and Medicare Paid is called **revenue leakage** —
the core business problem this entire project is designed to help analyse.

In [ ]:
print("=" * 55)
print("DATABASE SUMMARY — Texas Medicare Billing 2023")
print("=" * 55)

checks = [
    ("Total rows",              "SELECT COUNT(*) FROM cms_billing"),
    ("Unique providers",        "SELECT COUNT(DISTINCT Rndrng_NPI) FROM cms_billing"),
    ("Unique specialties",      "SELECT COUNT(DISTINCT Rndrng_Prvdr_Type) FROM cms_billing"),
    ("Unique cities",           "SELECT COUNT(DISTINCT Rndrng_Prvdr_City) FROM cms_billing"),
    ("Unique procedure codes",  "SELECT COUNT(DISTINCT HCPCS_Cd) FROM cms_billing"),
    ("Total billed ($)",        "SELECT ROUND(SUM(Tot_Sbmtd_Chrg),0) FROM cms_billing"),
    ("Total Medicare paid ($)", "SELECT ROUND(SUM(Tot_Mdcr_Pymt_Amt),0) FROM cms_billing"),
    ("Total leakage ($)",       "SELECT ROUND(SUM(Revenue_Leakage),0) FROM cms_billing"),
]

for label, query in checks:
    val = conn.execute(query).fetchone()[0]
    if isinstance(val, float):
        print(f"  {label:<28}  ${val:>20,.0f}")
    else:
        print(f"  {label:<28}  {val:>21,}")

print("\nTop 5 specialties by total billed:")
print(f"\n  {'Specialty':<35} {'Total Billed':>18} {'Medicare Paid':>18} {'Reimb %':>8}")
print("  " + "-" * 83)

rows = conn.execute("""
    SELECT
        Rndrng_Prvdr_Type,
        ROUND(SUM(Tot_Sbmtd_Chrg), 0)        AS total_billed,
        ROUND(SUM(Tot_Mdcr_Pymt_Amt), 0)     AS total_paid,
        ROUND(AVG(Reimbursement_Rate_Pct), 1) AS avg_reimb_pct
    FROM cms_billing
    GROUP BY Rndrng_Prvdr_Type
    ORDER BY total_billed DESC
    LIMIT 5
""").fetchall()

for row in rows:
    billed = f"${row[1]:>15,.0f}"
    paid   = f"${row[2]:>15,.0f}"
    print(f"  {str(row[0]):<35} {billed:>18} {paid:>18} {row[3]:>7}%")

conn.close()
print(f"\n✓ Database ready: {DB_PATH}")
print("  Next step: build agent.py")

In [ ]:
# Reopen the database connection
conn = sqlite3.connect(DB_PATH)
print("✓ Database connection reopened")

---
## Section 2 — Building the AI Agent

The database is ready. Now we build the Claude-powered layer that converts 
plain English questions into SQL queries against our healthcare billing data.

### How the agent works
Every question goes through 3 functions in sequence:

1. **generate_sql()** — sends the question + database schema to Claude → gets back SQL
2. **execute_sql()** — runs that SQL against cms_texas.db → gets back rows of data  
3. **generate_narrative()** — sends the results back to Claude → gets back a plain English insight

These 3 functions are then orchestrated by a 4th function called **run_agent()**
which chains them together and returns everything the UI needs in one call.

In [15]:
import anthropic

client = anthropic.Anthropic()

test = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=30,
    messages=[{
        "role": "user",
        "content": "Say: Claude is connected"
    }]
)

print("✓ Connected!")
print(f"  {test.content[0].text}")

✓ Connected!
  Claude is connected


### Schema Prompt
The schema prompt is injected into every Claude request as the system prompt.
It tells Claude exactly what our database contains before it writes any SQL.

This is called "context injection" — we give Claude a complete data dictionary
so it generates accurate SQL without hallucinating column names or wrong syntax.

The more precise the schema prompt, the more accurate Claude's SQL output.
This is the most important prompt engineering decision in the entire project.

In [16]:
# ── Schema Prompt ──────────────────────────────────────────────────────────────
# Injected into every Claude request as the system prompt.
# Claude reads this before answering any question.

SCHEMA_PROMPT = """
You are a healthcare business intelligence analyst assistant.
You have access to a SQLite database called cms_texas.db.

It contains one table called cms_billing with real Texas Medicare
billing data for 2023. The table has 671,400 rows — one row per
provider per procedure code.

TABLE: cms_billing

PROVIDER INFORMATION:
  Rndrng_NPI                 — Provider unique ID number (NPI)
  Rndrng_Prvdr_Last_Org_Name — Provider last name or organization name
  Rndrng_Prvdr_First_Name    — Provider first name
  Rndrng_Prvdr_Type          — Medical specialty
                               Examples: 'Cardiology', 'Internal Medicine',
                               'Ophthalmology', 'Diagnostic Radiology',
                               'Nurse Practitioner', 'Clinical Laboratory',
                               'Ambulatory Surgical Center'
  Rndrng_Prvdr_City          — City name
                               Examples: 'Houston', 'Dallas', 'San Antonio',
                               'Austin', 'Fort Worth', 'El Paso'
  Rndrng_Prvdr_State_Abrvtn  — Always 'TX' in this database
  Rndrng_Prvdr_Zip5          — 5-digit ZIP code

PROCEDURE INFORMATION:
  HCPCS_Cd                   — Procedure billing code (e.g. '99213')
  HCPCS_Desc                 — Plain English procedure name
                               e.g. 'Office/outpatient visit, established'
  Place_Of_Srvc              — Where service was performed
                               'F' = facility (hospital)
                               'O' = office

VOLUME COLUMNS:
  Tot_Benes                  — Total unique Medicare patients served
  Tot_Srvcs                  — Total number of services billed

AVERAGE DOLLAR COLUMNS (per service):
  Avg_Sbmtd_Chrg             — Average amount charged per service
  Avg_Mdcr_Alowd_Amt         — Average Medicare allowed per service
  Avg_Mdcr_Pymt_Amt          — Average Medicare payment per service

DERIVED TOTAL DOLLAR COLUMNS (average x Tot_Srvcs):
  Tot_Sbmtd_Chrg             — Total amount billed to Medicare
  Tot_Mdcr_Alowd_Amt         — Total amount Medicare allowed
  Tot_Mdcr_Pymt_Amt          — Total amount Medicare actually paid

REVENUE CYCLE METRICS (pre-calculated, ready to query directly):
  Reimbursement_Rate_Pct     — Percentage of billed amount Medicare paid
                               Formula: Tot_Mdcr_Pymt_Amt / Tot_Sbmtd_Chrg * 100
                               Low % = poor reimbursement
                               High % = strong reimbursement
  Revenue_Leakage            — Dollars billed but never collected
                               Formula: Tot_Sbmtd_Chrg - Tot_Mdcr_Pymt_Amt
                               Higher value = more money left uncollected

STRICT SQL RULES:
  1. This is SQLite — never use MySQL or PostgreSQL syntax
  2. Always alias aggregations: SUM(Tot_Sbmtd_Chrg) AS total_billed
  3. City filter always use LIKE: WHERE Rndrng_Prvdr_City LIKE '%Houston%'
  4. Specialty filter always use LIKE: WHERE Rndrng_Prvdr_Type LIKE '%Cardiology%'
  5. Always GROUP BY when using SUM() or AVG() across categories
  6. Always ORDER BY the main metric DESC unless asked otherwise
  7. Always LIMIT to 20 rows maximum unless a specific number is requested
  8. Round dollar amounts to 2 decimals: ROUND(SUM(Tot_Sbmtd_Chrg), 2)
  9. Round percentages to 1 decimal: ROUND(AVG(Reimbursement_Rate_Pct), 1)
  10. NEVER use DROP, DELETE, UPDATE, INSERT or ALTER — read only

BUSINESS CONTEXT:
  Revenue leakage = gap between what providers bill and what Medicare pays.
  Reimbursement rate = efficiency of collections. Industry average is 25-35%.
  Specialties below 20% reimbursement rate have significant collection problems.
  Ambulatory Surgical Centers and Diagnostic Radiology typically bill high
  but receive low reimbursement because Medicare pays a fixed fee schedule.
"""

print("✓ Schema prompt defined")
print(f"  Characters : {len(SCHEMA_PROMPT):,}")
print(f"  This is injected into every Claude request before the question")

✓ Schema prompt defined
  Characters : 3,835
  This is injected into every Claude request before the question


### generate_sql() Function
This function sends the user's plain English question to Claude
along with the schema prompt and gets back a structured JSON response
containing the SQL query, an explanation, and a recommended chart type.

Key design decisions:
- **Structured JSON output** — Claude returns JSON not free text, making
  parsing reliable and predictable every single time
- **Safety guardrail** — a Python check blocks any destructive SQL keywords
  before the query ever reaches the database
- **Chart type inference** — Claude recommends the right visualization based
  on the shape of the data, not a hardcoded rule

In [17]:
# ── Safety guardrail ───────────────────────────────────────────────────────────
# These keywords should never appear in a read-only analytics query.
# This is a second layer of protection on top of Claude's instructions.

BLOCKED_KEYWORDS = [
    "DROP", "DELETE", "INSERT", 
    "UPDATE", "ALTER", "TRUNCATE"
]

def is_safe_sql(sql: str) -> bool:
    """
    Returns False if the SQL contains any destructive keywords.
    Case-insensitive check — blocks 'drop', 'DROP', 'Drop' equally.
    """
    upper = sql.upper()
    return not any(keyword in upper for keyword in BLOCKED_KEYWORDS)


# ── generate_sql() ─────────────────────────────────────────────────────────────
# Sends the question + schema to Claude, gets back structured JSON.
# Returns a dictionary with sql, explanation, and chart_type.

def generate_sql(user_question: str) -> dict:
    """
    Convert a plain English question into a SQL query using Claude.
    
    Args:
        user_question: The question typed by the user
        
    Returns:
        dict with keys: sql, explanation, chart_type
    """
    
    # This is the instruction Claude receives along with the schema.
    # We tell it exactly what format to respond in — no ambiguity.
    instruction = """
Convert the user's question into a SQL query for the cms_billing table.

Respond ONLY with a valid JSON object in exactly this format — 
no text before it, no text after it, no markdown code fences:
{
  "sql": "SELECT ... FROM cms_billing ...",
  "explanation": "One plain sentence describing what this SQL finds",
  "chart_type": "bar"
}

For chart_type choose one of:
  "bar"   — comparing categories (specialties, cities, providers)
  "line"  — trends over time
  "pie"   — part of a whole (max 6 categories)
  "table" — detailed row-level data or more than 6 categories
"""
    
    # Send to Claude with the schema as system prompt
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        system=SCHEMA_PROMPT,    # database schema injected here
        messages=[
            {
                "role": "user", 
                "content": f"{instruction}\n\nUser question: {user_question}"
            }
        ]
    )
    
    # Extract the raw text response
    raw = response.content[0].text.strip()
    
    # Clean up in case Claude adds markdown code fences despite instructions
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    
    # Parse the JSON response
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        raise ValueError(f"Claude returned invalid JSON:\n{raw}")
    
    # Run the safety check before returning
    if not is_safe_sql(result["sql"]):
        raise ValueError(
            f"Generated SQL contains blocked keywords. Rejected."
        )
    
    return result


print("✓ generate_sql() defined")
print(f"  Blocked keywords : {BLOCKED_KEYWORDS}")
print(f"  Output format    : JSON with sql, explanation, chart_type")

✓ generate_sql() defined
  Blocked keywords : ['DROP', 'DELETE', 'INSERT', 'UPDATE', 'ALTER', 'TRUNCATE']
  Output format    : JSON with sql, explanation, chart_type


### execute_sql() Function
This function runs the Claude-generated SQL against cms_texas.db
and returns the results in a consistent structured format.

Key design decisions:
- **Try/except error handling** — SQL errors are caught and reported
  cleanly without crashing the agent
- **Consistent return format** — always returns columns, rows, and
  row_count so the UI always knows what shape the data is in
- **Connection management** — opens and closes the database connection
  within the function so connections are never left open accidentally

In [18]:
import sqlite3
import json
import re

DB_PATH = "cms_texas.db"

print(f"✓ DB_PATH set : {DB_PATH}")

✓ DB_PATH set : cms_texas.db


In [19]:
# ── execute_sql() ──────────────────────────────────────────────────────────────
# Runs the generated SQL against cms_texas.db and returns structured results.

def execute_sql(sql: str) -> dict:
    """
    Execute a SQL query against cms_texas.db.
    
    Args:
        sql: The SQL query string to execute
        
    Returns:
        dict with keys:
          columns  — list of column names
          rows     — list of rows, each row is a list of values
          row_count — total number of rows returned
    """
    
    # Open a fresh connection for this query
    conn = sqlite3.connect(DB_PATH)
    
    # row_factory makes rows accessible by column name
    conn.row_factory = sqlite3.Row
    
    cur = conn.cursor()
    
    try:
        # Execute the query
        cur.execute(sql)
        
        # Fetch all results
        raw_rows = cur.fetchall()
        
        # Extract column names from the cursor description
        # cur.description contains metadata about each column returned
        columns = [desc[0] for desc in cur.description] if cur.description else []
        
        # Convert each row from sqlite3.Row to a plain Python list
        # This makes it easy to serialize to JSON later for the UI
        rows = [list(row) for row in raw_rows]
        
    except sqlite3.Error as e:
        # Close connection before raising the error
        conn.close()
        raise ValueError(
            f"SQL execution failed: {e}\n\n"
            f"The query that failed was:\n{sql}"
        )
    
    # Always close the connection when done
    conn.close()
    
    return {
        "columns"  : columns,
        "rows"     : rows,
        "row_count": len(rows)
    }


print("✓ execute_sql() defined")
print(f"  Database  : {DB_PATH}")
print(f"  Returns   : columns, rows, row_count")
print(f"  Error handling: SQL errors caught and reported cleanly")

✓ execute_sql() defined
  Database  : cms_texas.db
  Returns   : columns, rows, row_count
  Error handling: SQL errors caught and reported cleanly


### generate_narrative() Function
After the SQL runs and returns data, this function sends the results
back to Claude to generate a plain English executive summary.

This is what makes the agent feel intelligent rather than just mechanical.
Instead of showing raw numbers, it produces a 2-3 sentence insight that
highlights the key finding, any notable outliers, and one actionable takeaway.

This mirrors what a real BI analyst does before presenting data to leadership —
they don't just show the table, they tell the story behind the numbers.

In [20]:
# ── generate_narrative() ───────────────────────────────────────────────────────
# Sends query results back to Claude to generate a plain English summary.
# This is what makes the agent feel like a real analyst, not just a SQL runner.

def generate_narrative(
    user_question : str,
    sql           : str,
    query_result  : dict
) -> str:
    """
    Generate a plain English executive summary of the query results.
    
    Args:
        user_question : The original question the user asked
        sql           : The SQL query that was executed
        query_result  : The dict returned by execute_sql()
        
    Returns:
        A 2-3 sentence executive summary string
    """
    
    # Only send the first 10 rows to Claude — enough for a good summary
    # without wasting tokens on rows that don't change the insight
    preview_rows = query_result["rows"][:10]
    
    result_preview = {
        "columns"    : query_result["columns"],
        "sample_rows": preview_rows,
        "total_rows" : query_result["row_count"]
    }
    
    prompt = f"""
The user asked this question about Texas Medicare billing data:
"{user_question}"

We ran this SQL query:
{sql}

The results ({len(preview_rows)} of {query_result['row_count']} total rows):
{json.dumps(result_preview, indent=2)}

Write a concise 2-3 sentence executive summary of these results.

Your summary must:
1. State the single most important finding with specific numbers
2. Highlight any notable outlier or pattern worth attention
3. End with one actionable business takeaway

Write in plain business language — no SQL, no technical jargon.
Write as if presenting to a hospital CFO or revenue cycle director.
Do NOT start with "The query" or "Based on the data" or "The results show".
"""
    
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.content[0].text.strip()


print("✓ generate_narrative() defined")
print(f"  Model      : claude-sonnet-4-6")
print(f"  Max tokens : 300 (enough for 2-3 sentences)")
print(f"  Sends      : first 10 rows + column names + total row count")
print(f"  Returns    : plain English executive summary string")

✓ generate_narrative() defined
  Model      : claude-sonnet-4-6
  Max tokens : 300 (enough for 2-3 sentences)
  Sends      : first 10 rows + column names + total row count
  Returns    : plain English executive summary string


### run_agent() Orchestrator
This is the main function that chains all three previous functions together.
The UI calls only this function — it handles everything internally.

Flow:
  1. generate_sql()       → converts question to SQL + chart type
  2. execute_sql()        → runs SQL against cms_texas.db
  3. generate_narrative() → writes plain English summary of results
  4. packages everything  → ret

In [21]:
# ── run_agent() ────────────────────────────────────────────────────────────────
# The main orchestrator. Chains generate_sql → execute_sql → generate_narrative.
# This is the only function the UI needs to call.

def run_agent(user_question: str) -> dict:
    """
    Full pipeline: plain English question → SQL → results → narrative.
    
    Args:
        user_question: The question typed by the user in plain English
        
    Returns:
        dict containing everything the UI needs to display:
          question    — the original question
          sql         — the generated SQL query
          explanation — plain English explanation of what the SQL does
          chart_type  — recommended chart type (bar/line/pie/table)
          columns     — list of column names
          rows        — list of data rows
          row_count   — total number of rows returned
          narrative   — AI-generated executive summary
    """
    
    print(f"\n{'='*60}")
    print(f"Question: {user_question}")
    print(f"{'='*60}")
    
    # ── Step 1: Generate SQL ───────────────────────────────────────
    print("\n[Step 1] Sending question to Claude...")
    sql_result = generate_sql(user_question)
    
    print(f"[Step 1] SQL generated:")
    print(f"         {sql_result['sql'][:100]}...")  # print first 100 chars
    print(f"[Step 1] Chart type: {sql_result['chart_type']}")
    
    # ── Step 2: Execute SQL ────────────────────────────────────────
    print(f"\n[Step 2] Running SQL against {DB_PATH}...")
    query_result = execute_sql(sql_result["sql"])
    
    print(f"[Step 2] Returned {query_result['row_count']} rows")
    print(f"[Step 2] Columns: {query_result['columns']}")
    
    # ── Step 3: Generate narrative ─────────────────────────────────
    print(f"\n[Step 3] Generating executive summary...")
    narrative = generate_narrative(
        user_question,
        sql_result["sql"],
        query_result
    )
    
    print(f"[Step 3] Narrative:")
    print(f"         {narrative[:150]}...")  # print first 150 chars
    
    # ── Package everything for the UI ─────────────────────────────
    return {
        "question"   : user_question,
        "sql"        : sql_result["sql"],
        "explanation": sql_result["explanation"],
        "chart_type" : sql_result["chart_type"],
        "columns"    : query_result["columns"],
        "rows"       : query_result["rows"],
        "row_count"  : query_result["row_count"],
        "narrative"  : narrative
    }


print("✓ run_agent() defined")
print(f"  Input  : plain English question (string)")
print(f"  Output : dict with sql, chart_type, rows, narrative")
print(f"  Steps  : generate_sql → execute_sql → generate_narrative")

✓ run_agent() defined
  Input  : plain English question (string)
  Output : dict with sql, chart_type, rows, narrative
  Steps  : generate_sql → execute_sql → generate_narrative


---
## Section 3 — Live Demo Queries

The agent is fully built. These three queries demonstrate the complete pipeline
against real Texas Medicare billing data.

Each query is chosen to tell a different part of the revenue cycle story:
- Query 1 — Specialty level: which specialties lose the most money
- Query 2 — City level: how Houston compares in reimbursement
- Query 3 — Provider level: which individual providers bill the most

These are the three questions to run during any portfolio demo or interview.

### Query 1 — Which specialties have the worst reimbursement rates in Texas?
This is the headline revenue leakage story from our dataset.
Low reimbursement rate = significant gap between what was billed and what was collected.

In [12]:
result1 = run_agent(
    "Which medical specialties have the worst reimbursement rates in Texas? "
    "Show the top 10 with total billed, total paid, and reimbursement rate."
)

print("\n── RESULT ──────────────────────────────────────────")
print(f"Question   : {result1['question']}")
print(f"Chart type : {result1['chart_type']}")
print(f"Rows       : {result1['row_count']}")
print(f"\nSQL:\n{result1['sql']}")
print(f"\nNarrative:\n{result1['narrative']}")


Question: Which medical specialties have the worst reimbursement rates in Texas? Show the top 10 with total billed, total paid, and reimbursement rate.

[Step 1] Sending question to Claude...
[Step 1] SQL generated:
         SELECT Rndrng_Prvdr_Type AS specialty, ROUND(SUM(Tot_Sbmtd_Chrg), 2) AS total_billed, ROUND(SUM(Tot_...
[Step 1] Chart type: bar

[Step 2] Running SQL against cms_texas.db...
[Step 2] Returned 10 rows
[Step 2] Columns: ['specialty', 'total_billed', 'total_paid', 'avg_reimbursement_rate_pct']

[Step 3] Generating executive summary...
[Step 3] Narrative:
         Anesthesiology-related specialties dominate the bottom of the reimbursement spectrum in Texas, with Anesthesiology Assistants receiving just **6.9 cen...

── RESULT ──────────────────────────────────────────
Question   : Which medical specialties have the worst reimbursement rates in Texas? Show the top 10 with total billed, total paid, and reimbursement rate.
Chart type : bar
Rows       : 10

SQL:
SELECT R

### Query 2 — Houston provider revenue leakage
Zooming into Houston specifically — relevant for local hiring managers
who work with Houston health systems like Houston Methodist, MD Anderson,
or Memorial Hermann.

In [13]:
result2 = run_agent(
    "Which providers in Houston have the highest total revenue leakage? "
    "Show the top 10 providers with their specialty, total billed, "
    "total Medicare paid, and revenue leakage amount."
)

print("\n── RESULT ──────────────────────────────────────────")
print(f"Question   : {result2['question']}")
print(f"Chart type : {result2['chart_type']}")
print(f"Rows       : {result2['row_count']}")
print(f"\nSQL:\n{result2['sql']}")
print(f"\nNarrative:\n{result2['narrative']}")


Question: Which providers in Houston have the highest total revenue leakage? Show the top 10 providers with their specialty, total billed, total Medicare paid, and revenue leakage amount.

[Step 1] Sending question to Claude...
[Step 1] SQL generated:
         SELECT Rndrng_Prvdr_Last_Org_Name, Rndrng_Prvdr_First_Name, Rndrng_Prvdr_Type, ROUND(SUM(Tot_Sbmtd_C...
[Step 1] Chart type: bar

[Step 2] Running SQL against cms_texas.db...
[Step 2] Returned 10 rows
[Step 2] Columns: ['Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_First_Name', 'Rndrng_Prvdr_Type', 'total_billed', 'total_medicare_paid', 'total_revenue_leakage']

[Step 3] Generating executive summary...
[Step 3] Narrative:
         **Revenue leakage among Houston's top 10 providers totals approximately $670 million**, with Quest Diagnostics and Boston Scientific alone accounting ...

── RESULT ──────────────────────────────────────────
Question   : Which providers in Houston have the highest total revenue leakage? Show the top 10 p

### Query 3 — Highest billed procedures in Texas
Procedure-level analysis shows which specific services drive the most
billing volume — useful for understanding where revenue cycle 
optimization efforts should be focused.

In [14]:
result3 = run_agent(
    "What are the top 10 most billed procedures in Texas by total charges? "
    "Show the procedure description, total services, total billed, "
    "and average reimbursement rate."
)

print("\n── RESULT ──────────────────────────────────────────")
print(f"Question   : {result3['question']}")
print(f"Chart type : {result3['chart_type']}")
print(f"Rows       : {result3['row_count']}")
print(f"\nSQL:\n{result3['sql']}")
print(f"\nNarrative:\n{result3['narrative']}")


Question: What are the top 10 most billed procedures in Texas by total charges? Show the procedure description, total services, total billed, and average reimbursement rate.

[Step 1] Sending question to Claude...
[Step 1] SQL generated:
         SELECT HCPCS_Desc, SUM(Tot_Srvcs) AS total_services, ROUND(SUM(Tot_Sbmtd_Chrg), 2) AS total_billed, ...
[Step 1] Chart type: bar

[Step 2] Running SQL against cms_texas.db...
[Step 2] Returned 10 rows
[Step 2] Columns: ['HCPCS_Desc', 'total_services', 'total_billed', 'avg_reimbursement_rate']

[Step 3] Generating executive summary...
[Step 3] Narrative:
         **Office visits dominate Texas Medicare billing**, with established patient outpatient visits (30-39 minutes) alone generating nearly **$1.8 billion i...

── RESULT ──────────────────────────────────────────
Question   : What are the top 10 most billed procedures in Texas by total charges? Show the procedure description, total services, total billed, and average reimbursement rate.
Ch